In [1]:
import os
import json

In [2]:
from cimgraph.databases.fileparsers import XMLFile
from cimgraph.databases import ConnectionParameters
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim 
from uuid import UUID

In [3]:
params = ConnectionParameters(filename="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13/IEEE13.xml", cim_profile='cimhub_2023', iec61970_301=8)
xml_file = XMLFile(params)
feeder_mrid = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
feeder = cim.Feeder(mRID = feeder_mrid, name = 'ieee13')
feeder_model = FeederModel(connection=xml_file, container=feeder, distributed=False)

In [4]:
from cimtopology.distribution import create_distributed_feeder

create_distributed_feeder(feeder_model, 'ieee13_distributed.xml')

In [5]:
from cimgraph.databases import BlazegraphConnection
from cimloader.databases import BlazegraphConnection as BG
from cimloader.uploaders import BlazegraphUploader

params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql", cim_profile='cimhub_2023', iec61970_301=8)
bg = BG(params)
loader = BlazegraphUploader(params)
blazegraph = BlazegraphConnection(params)
bg.drop_all()

loader.upload_from_file(filepath='./', filename='ieee13_distributed.xml')
# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="IEEE13.xml")

# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="measurements.xml")

HTTP/1.1 100 Continue

HTTP/1.1 200 OK
Content-Type: application/xml;charset=iso-8859-1
Content-Length: 62
Server: Jetty(9.4.18.v20190429)

<?xml version="1.0"?><data modified="4829" milliseconds="38"/>

In [6]:
from gridappsd import GridAPPSD
os.environ['GRIDAPPSD_USER'] = 'app_user'
os.environ['GRIDAPPSD_PASSWORD'] = '1234App'
gapps = GridAPPSD()
assert gapps.connected


In [7]:
topic = "goss.gridappsd.request.data.cimtopology2"

message = {
   "requestType": "GET_DISTRIBUTED_AREAS",
   "mRID":  "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62",
   "resultFormat": "JSON"
}

topo_message = gapps.get_response(topic, message, timeout=30)

print(json.dumps(topo_message, indent=4))


{
    "DistributionArea": {
        "mRID": "94DFDC1C-8B80-4E90-96E1-504305EDDCC2",
        "Substations": [
            "6C62C905-6FC7-653D-9F1E-1340F974A587"
        ],
        "Feeders": [
            "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
        ],
        "FeederAreas": [
            {
                "mRID": "B6373777-16CB-45D7-9BF0-6F51D91CF949",
                "Feeder": "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62",
                "AddressableEquipment": [
                    {
                        "@id": "52DE9189-20DC-4C73-BDEE-E960FE1F9493",
                        "@type": "Breaker"
                    },
                    {
                        "@id": "CE5D0651-676B-4AF3-8D67-41BF1B33E30C",
                        "@type": "Recloser"
                    },
                    {
                        "@id": "2858B6C2-0886-4269-884C-06FA8B887319",
                        "@type": "LoadBreakSwitch"
                    },
                    {
                        "@i

In [10]:
with open('ieee13_topo_msg.json','w') as file:
    json.dump(topo_message, file, indent = 4)

In [8]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['AddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

AttributeError: 'NoneType' object has no attribute 'identifier'

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['UnaddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['Measurements']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [3]:
from uuid import UUID
jsonld = dict["@id":UUID,"@type":str(type)]


In [ ]:
def fun(c:jsonld) -> jsonld:
    print(c)

fun('hello')